**Project:Disease Prediction Model**

**Overview**

This project aims to predict possible diseases based on user-reported symptoms and recommend the appropriate type of doctor for consultation. Such a system can assist in preliminary health assessment and guide users toward the right medical specialist. The dataset contains numerous symptoms represented as binary features along with their associated diseases and doctor types. An XGBoost multi-class classification model was trained to learn the relationship between symptom patterns and diseases. The system allows users to select symptoms through an interactive interface and returns the top three most probable diseases along with the recommended doctor. This project demonstrates how machine learning can be applied to healthcare data to support early disease identification and improve decision-making.

**1. Importing Required Libraries**

Install the XGBoost library, which is the machine learning algorithm used in the project and interactive widgets for Colab.

In [ ]:
!pip install xgboost ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.7 MB/s eta 0:00:00


Here, we import essential libraries:
1. Pandas: Used for data manipulation and analysis. It provides powerful data structures like DataFrames to efficiently handle tabular datasets.
2. NumPy: Supports numerical operations and array processing, enabling efficient mathematical computations.
3. Scikit-learn Modules:
*   train_test_split: Used to divide the dataset into training and testing sets, allowing the model to be trained on one portion of the data and evaluated on unseen data.
*   LabelEncoder: Converts categorical disease labels into numerical values so that they can be processed by machine learning algorithms.
*   accuracy_score: Used to evaluate the performance of the trained model by calculating the percentage of correct predictions.
4. XGBoost: XGBoost is a powerful gradient boosting machine learning algorithm used for classification tasks. In this project, it is used to learn the relationship between symptoms and diseases and predict the most probable disease based on selected symptoms.
5. ipywidgets: Used to create interactive user interface elements such as dropdown menus and buttons inside a Google Colab environment. It allows users to select symptoms easily instead of manually entering values.
6. IPython Display: The display function is used to render interactive widgets and output elements within the notebook interface, enabling the interactive symptom selection and prediction display.









In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
import ipywidgets as widgets
from IPython.display import display

**2. Loading the Dataset**

files.upload() is used to upload a file from the local machine to the Google Colab environment. A file picker dialog appears, allowing you to select the dataset (in this case, data.csv).
The uploaded file is temporarily stored in the Colab session and is accessible by its filename.

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Augmented_Dataset_Reduced_50.csv to Augmented_Dataset_Reduced_50 (2).csv


**3. Data Preprocessing**

The pd.read_csv function reads the uploaded CSV file and converts it into a Pandas DataFrame. While the head() function displays the first 5 rows of the dataset to understand its structure and content.

In [ ]:
df = pd.read_csv("Augmented_Dataset_Reduced_50.csv")
df.head()

,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,problems with orgasm,nose deformity,lump over jaw,sore in nose,hip weakness,back swelling,ankle stiffness or tightness,ankle weakness,neck weakness,doctor_type
0,chronic glaucoma,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Ophthalmologist
1,torticollis,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,Neurologist
2,personality disorder,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,Psychiatrist
3,fracture of the patella,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Orthopedic Surgeon
4,chronic otitis media,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Otolaryngologist


Remove diseases that appear very few times in the dataset.

In [ ]:
df = df.groupby("diseases").filter(lambda x: len(x) >= 10)

Separate the dataset into input features and target variables for training the machine learning model. Create the feature set by removing the disease and doctor columns, leaving only the symptom columns that will be used as inputs to the model. Stores the disease labels, which are the values the model will learn to predict and saves the type of doctor associated with each disease, which will later be used to recommend the appropriate doctor after the model predicts the disease.

In [ ]:
X = df.drop(["diseases","doctor_type"], axis=1)
Y = df["diseases"]
doctor = df["doctor_type"]

**4. Model Training,Evaluation and Testing**

Convert the disease names, which are in text form, into numerical values so that the machine learning model can process them. Each disease is assigned a unique number, allowing the model to work with the data since most machine learning algorithms require numerical inputs instead of text labels.

In [ ]:
le = LabelEncoder()
Y_encoded = le.fit_transform(Y)

Split the dataset into training and testing sets. The training set is used to teach the model the relationship between symptoms and diseases, while the testing set is used to evaluate how well the trained model performs on unseen data. Here, 20% of the data is reserved for testing, and the rest is used for training. The stratify parameter ensures that the proportion of each disease class remains similar in both the training and testing datasets, leading to a more balanced and reliable evaluation.

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=Y_encoded
)

Create and train the XGBoost classification model for predicting diseases based on symptoms. The model is configured with several parameters that control how the decision trees are built, such as the number of trees, their depth, and the learning rate, which together help improve prediction performance. After defining the model, it is trained using the training dataset, allowing it to learn patterns between the symptoms and the corresponding diseases.

In [ ]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    tree_method="hist",
    n_jobs=-1,
    objective="multi:softprob",
    eval_metric="mlogloss"
)

model.fit(X_train, Y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=8, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=-1,
              num_parallel_tree=None, ...)

Evaluate the performance of the trained model. The model first makes predictions on the testing dataset, which contains data it has not seen during training. These predicted values are then compared with the actual disease labels in the test set to calculate the accuracy score. The accuracy represents the proportion of correct predictions made by the model and provides an indication of how well the model performs on unseen data.

In [ ]:
pred = model.predict(X_test)
accuracy = accuracy_score(Y_test, pred)
print("Model Accuracy:", accuracy)

Model Accuracy: 0.8355935654652821


**5. Widget Creation**

Create a mapping between diseases and their corresponding doctor types so the correct doctor can be recommended after prediction. Prepared the list of symptom names for the user interface by converting underscores to spaces and sorting them alphabetically. Finally create mapping so that the user-selected symptom names can be correctly matched with the dataset's original symptom columns

In [ ]:
doctor_map = df.groupby("diseases")["doctor_type"].first().to_dict()

In [ ]:
symptom_columns = list(X.columns)

display_symptoms = sorted([s.replace("_"," ") for s in symptom_columns])

symptom_to_column = {s.replace("_"," "): s for s in symptom_columns}

The ipywidgets library is used to build UI components such as dropdown menus and buttons, while display is used to show these elements in the notebook. A dropdown menu is then created that contains the list of symptoms, allowing the user to easily select a symptom from the available options.

Then created the interactive buttons and display elements for the user interface. Three buttons are defined to allow the user to add a symptom, remove the last selected symptom, and predict the disease. A list is also created to store the symptoms selected by the user. Additionally, display elements are set up to show the selected symptoms and the prediction results within the notebook interface.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

In [ ]:
symptom_dropdown = widgets.Dropdown(
    options=display_symptoms,
    description="Symptom:"
)

In [ ]:
add_button = widgets.Button(description="Add Symptom", button_style='success')
remove_button = widgets.Button(description="Remove Last", button_style='warning')
predict_button = widgets.Button(description="Predict Disease", button_style='primary')

In [ ]:
selected_symptoms = []
symptom_display = widgets.HTML()
output = widgets.Output()

Update the display of selected symptoms in the interface. If no symptoms have been selected, it shows a message indicating that no symptoms are selected. Otherwise, it displays the list of symptoms that the user has chosen, making it easier to keep track of the selected inputs before running the prediction.

In [ ]:
def update_display():
    if len(selected_symptoms) == 0:
        symptom_display.value = "<b>No symptoms selected</b>"
    else:
        symptom_display.value = "<b>Selected Symptoms:</b><br>" + "<br>".join(selected_symptoms)

Allow the user to add a symptom from the dropdown menu to the selected symptoms list. When the Add Symptom button is clicked, the chosen symptom is added to the list if it is not already present. After adding the symptom, the display is updated so the user can see the currently selected symptoms.

In [ ]:
def add_symptom(b):
    symptom = symptom_dropdown.value
    if symptom not in selected_symptoms:
        selected_symptoms.append(symptom)
    update_display()
add_button.on_click(add_symptom)

Allow the user to remove the most recently added symptom from the selected symptoms list. When the Remove Last button is clicked, the last symptom in the list is removed if any symptoms are present, and the display is updated to reflect the change.

In [ ]:
def remove_symptom(b):
    if selected_symptoms:
        selected_symptoms.pop()
    update_display()
remove_button.on_click(remove_symptom)

Perform the disease prediction when the Predict button is clicked. It first checks whether at least two symptoms have been selected. The selected symptoms are then converted into a numerical input vector that matches the format used during model training. The trained model predicts the probability of each disease, and the top three most probable diseases are displayed along with their recommended doctor types and prediction probabilities.

In [ ]:
def predict_disease(b):
    with output:
        output.clear_output()
        if len(selected_symptoms) < 2:
            print("Please select at least 2 symptoms")
            return
        input_vector = [0] * len(symptom_columns)
        for symptom in selected_symptoms:
            column = symptom_to_column[symptom]
            index = symptom_columns.index(column)
            input_vector[index] = 1
        input_data = np.array(input_vector).reshape(1,-1)
        probabilities = model.predict_proba(input_data)
        top3 = probabilities[0].argsort()[-3:][::-1]
        diseases = le.inverse_transform(top3)
        print("Top 3 Predicted Diseases:\n")
        for i,d in enumerate(diseases,1):
            prob = probabilities[0][top3[i-1]] * 100
            print(f"{i}. {d} → {doctor_map[d]} ({prob:.2f}%)")
predict_button.on_click(predict_disease)

**6. Output**

In [ ]:
display(
    symptom_dropdown,
    add_button,
    remove_button,
    symptom_display,
    predict_button,
    output
)

Dropdown(description='Symptom:', options=('abdominal distention', 'abnormal appearing skin', 'abnormal appeari…

Button(button_style='success', description='Add Symptom', style=ButtonStyle())

Button(button_style='warning', description='Remove Last', style=ButtonStyle())

HTML(value='')

Button(button_style='primary', description='Predict Disease', style=ButtonStyle())

Output()

**7. Conclusion**

This project demonstrates how machine learning can be used to predict possible diseases based on user-selected symptoms and recommend the appropriate type of doctor. Using an XGBoost classification model, the system learns patterns between symptoms and diseases and provides the top three most probable predictions. The interactive interface allows users to easily select symptoms, making the system practical and user-friendly. Overall, the project highlights the potential of data-driven approaches in supporting preliminary healthcare decision-making.